In [ ]:
import random
import string
import pandas as pd
import numpy as np
from datasets import load_dataset
import json
from huggingface_hub import login
import os
import matplotlib.pyplot as plt
from PIL import Image
import math
import stanza

from vqa_evaluation_prompts import *
from vqa_evaluator import VQAEvaluator
vqa_evaluator = VQAEvaluator()

# Raw output files relies on manual post-precessing
PREDICTION_FOLDER_PATH = "gemini_5_shot"
LABEL_FOLDER_PATH = "../../Annotations"
IMAGE_FOLDER = "../../10k_images"
print(PREDICTION_FOLDER_PATH)
print(LABEL_FOLDER_PATH)
print(IMAGE_FOLDER)

label_file = os.path.join(LABEL_FOLDER_PATH, "random1_vqa_references.json")
prediction_file = os.path.join(PREDICTION_FOLDER_PATH, "gemini_5_shot_random1_vqa_seed1.json")

/home/xuezheng/anaconda3/envs/evaluations/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


results/gemini_5_shot
../../Annotations
../../10k_images


# Gemini 5 shot

## Multi-label classification

In [3]:
y_pred, y_true, _ = vqa_evaluator.multilabel_classification_score(prediction_file, label_file)

0000001
{'0': 'No violations'}
0000002
{'1': {'reason': 'Multiple workers on the embankment are not wearing hard hats.', 'bounding_box': [0.38, 0.65, 0.49, 0.49]}, '4': {'reason': 'Multiple workers are within the operation radius of the excavator with an operator inside.', 'bounding_box': [0.38, 0.65, 0.49, 0.49]}}
0000005
{'1': {'reason': 'The worker on the left is not wearing a hard hat or high-visibility vest.', 'bounding_box': [0.22, 0.44, 0.35, 0.49]}}
0000007
{'1': {'reason': 'Multiple workers are not wearing hard hats or high-visibility vests while working at night. Specifically, the worker in the white shirt and black pants, the worker in the camouflage jacket, and the worker in the dark jacket on the right are not wearing hard hats. Additionally, none of the workers are wearing high-visibility vests, which are required for night work.', 'bounding_box': [0.0, 0.0, 1.0, 1.0]}, '3': {'reason': 'There are unprotected openings on both the left and right sides of the image. These op

## Bounding box

In [ ]:
# Run this evaluation first, the correctly-predicted-file generated will be used for the evaluation of explanation
IoU = vqa_evaluator.bounding_box_score(prediction_file, label_file)
print(IoU)

0000001
{'0': 'No violations'}
0000002
{'1': {'reason': 'Multiple workers on the embankment are not wearing hard hats.', 'bounding_box': [0.38, 0.65, 0.49, 0.49]}, '4': {'reason': 'Multiple workers are within the operation radius of the excavator with an operator inside.', 'bounding_box': [0.38, 0.65, 0.49, 0.49]}}
0000005
{'1': {'reason': 'The worker on the left is not wearing a hard hat or high-visibility vest.', 'bounding_box': [0.22, 0.44, 0.35, 0.49]}}
0000007
{'1': {'reason': 'Multiple workers are not wearing hard hats or high-visibility vests while working at night. Specifically, the worker in the white shirt and black pants, the worker in the camouflage jacket, and the worker in the dark jacket on the right are not wearing hard hats. Additionally, none of the workers are wearing high-visibility vests, which are required for night work.', 'bounding_box': [0.0, 0.0, 1.0, 1.0]}, '3': {'reason': 'There are unprotected openings on both the left and right sides of the image. These op

## Explanation

In [ ]:
# Get the number of correctly predicted rules
correctly_predicted = vqa_evaluator.correctly_predicted_images(prediction_file, label_file, isGeneratedFile=True)
print(correctly_predicted)
print(len(correctly_predicted['rule1']))
print(len(correctly_predicted['rule2']))
print(len(correctly_predicted['rule3']))
print(len(correctly_predicted['rule4']))

0000001
{'0': 'No violations'}
0000002
{'1': {'reason': 'Multiple workers on the embankment are not wearing hard hats.', 'bounding_box': [0.38, 0.65, 0.49, 0.49]}, '4': {'reason': 'Multiple workers are within the operation radius of the excavator with an operator inside.', 'bounding_box': [0.38, 0.65, 0.49, 0.49]}}
0000005
{'1': {'reason': 'The worker on the left is not wearing a hard hat or high-visibility vest.', 'bounding_box': [0.22, 0.44, 0.35, 0.49]}}
0000007
{'1': {'reason': 'Multiple workers are not wearing hard hats or high-visibility vests while working at night. Specifically, the worker in the white shirt and black pants, the worker in the camouflage jacket, and the worker in the dark jacket on the right are not wearing hard hats. Additionally, none of the workers are wearing high-visibility vests, which are required for night work.', 'bounding_box': [0.0, 0.0, 1.0, 1.0]}, '3': {'reason': 'There are unprotected openings on both the left and right sides of the image. These op

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, Conversation, BitsAndBytesConfig, set_seed
from evaluate import load
import torch
import json
import os
import ast
import random
from vqa_evaluation_prompts import *

set_seed(20)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quantization_config, device_map="auto", num_beams = 5)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quantization_config, device_map="auto")

/home/xuezheng/anaconda3/envs/llama3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:03<00:00,  1.32it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
prompt_dict = {
    "system_prompt": [SYSTEM_PROMPT_RULE_1, SYSTEM_PROMPT_RULE_2, SYSTEM_PROMPT_RULE_3, SYSTEM_PROMPT_RULE_4],
    "user_prompt": [FEW_SHOT_PROMPT, USER_PROMPT_FINAL],
    "example_prompt": [[EXAMPLE_EVAL_RULE1_PROMPT_0000545, EXAMPLE_EVAL_RULE1_PROMPT_0000007, EXAMPLE_EVAL_RULE1_PROMPT_0000019], 
                       [EXAMPLE_EVAL_RULE2_PROMPT_0000925, EXAMPLE_EVAL_RULE2_PROMPT_0003632, EXAMPLE_EVAL_RULE2_PROMPT_0004235], 
                       [EXAMPLE_EVAL_RULE3_PROMPT_0001597, EXAMPLE_EVAL_RULE3_PROMPT_0000007, EXAMPLE_EVAL_RULE3_PROMPT_0000117], 
                       [EXAMPLE_EVAL_RULE4_PROMPT_0001512, EXAMPLE_EVAL_RULE4_PROMPT_0004725, EXAMPLE_EVAL_RULE4_PROMPT_0002093]]
}
folder_to_evaluate = "Gemini_5_shot_correctly_predicted"
all_files = os.listdir(folder_to_evaluate)
all_files = sorted(all_files)
prediction_files = [os.path.join(folder_to_evaluate, file) for file in all_files if file.endswith('.json')]

final_mark_dict = {}
i = 0
for file in prediction_files:
    print(file)
    mark_dict = {}
    
    with open(file, 'r') as infile:
        correctly_predicted = json.load(infile)
    id_list = sorted(list(correctly_predicted.keys()))

    for instance in id_list:
        print(instance)

        chatbot = pipeline(task="conversational", model=model, tokenizer=tokenizer)

        conversation = Conversation([{"role": "system", "content": prompt_dict['system_prompt'][i]}])
        conversation = chatbot(conversation)
        conversation.add_message({"role": "user", "content": prompt_dict['user_prompt'][0]})
        conversation = chatbot(conversation)

        conversation.add_message({"role": "user", "content": prompt_dict['example_prompt'][i][0]})
        conversation = chatbot(conversation)
        conversation.add_message({"role": "user", "content": prompt_dict['example_prompt'][i][1]})
        conversation = chatbot(conversation)
        conversation.add_message({"role": "user", "content": prompt_dict['example_prompt'][i][2]})
        conversation = chatbot(conversation)

        conversation.add_message({"role": "user", "content": f"Please evaluate the following reasoning:\n\n Candidate reasoning: {correctly_predicted[instance]['candidate']}\n\n Reference reasoning{correctly_predicted[instance]['reference']}"})
        conversation = chatbot(conversation)
        reply = conversation.messages[-1]["content"]
        print(correctly_predicted[instance]['candidate'])
        print(correctly_predicted[instance]['reference'])

        conversation.add_message({"role": "user", "content": prompt_dict['user_prompt'][1]})
        conversation = chatbot(conversation)
        llama_eval = conversation.messages[-1]["content"]
        llama_eval_dict= ast.literal_eval(llama_eval)
        print(llama_eval_dict)

        mark_dict[instance] = llama_eval_dict

    final_mark_dict[f"rule{i+1}"] = mark_dict

    i += 1

for rule_id, eval_dict in final_mark_dict.items():
    print(f"Average for {rule_id}:")

    mark_list = []
    for image_id, marks in eval_dict.items():
        mark_list.append(marks['total'])
    print(sum(mark_list)/len(mark_list))
    print(f"Number of instance: {len(mark_list)}")

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Gemini_5_shot_correctly_predicted/rule1_correctly_predicted.json
0000005


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end gene

The worker on the left is not wearing a hard hat or high-visibility vest.
Person on the left not using PPE.
{'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}
0000007
Multiple workers are not wearing hard hats or high-visibility vests while working at night. Specifically, the worker in the white shirt and black pants, the worker in the camouflage jacket, and the worker in the dark jacket on the right are not wearing hard hats. Additionally, none of the workers are wearing high-visibility vests, which are required for night work.
Multiple workers not wearing hard hats nor high-visibility vests.
{'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}
0000009
The worker in the orange jacket is not wearing a hard hat. The worker in the blue shirt is not wearing a hard hat.
Workers next to the truck on the left not wearing high visibility vests.
{'relevance': 2, 'equivalence': 0, 'specificity': 1, 'total': 3}
0000019
Worker with a black cap and white shirt on the left